## Replicando o tratamento de dados do primeiro, segundo e terceiro arquivo.

In [1]:
# Importando o pandas
import pandas as pd

In [2]:
# Visualizando a base de treino
treino = pd.read_csv('train.csv')
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [3]:
# Visualizando a base de teste
teste = pd.read_csv('test.csv')
teste.head(3)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q


In [4]:
# Eliminando as colunas com elevada cardinalidade
treino = treino.drop(['Name','Ticket','Cabin'],axis=1)
teste = teste.drop(['Name','Ticket','Cabin'],axis=1)

In [5]:
# Usando a média para substituir valores nulos na coluna de idade
treino.loc[treino.Age.isnull(),'Age'] = treino.Age.mean()
teste.loc[teste.Age.isnull(),'Age'] = teste.Age.mean()

In [6]:
# Tratando a coluna Embarked da base de treino usando a moda 
treino.loc[treino.Embarked.isnull(),'Embarked'] = treino.Embarked.mode()[0]

In [7]:
# E também a coluna Fare da base de teste usando a média
teste.loc[teste.Fare.isnull(),'Fare'] = teste.Fare.mean()

In [8]:
# Usando uma lambda function para tratar a coluna "Sex"
treino['MaleCheck'] = treino.Sex.apply(lambda x: 1 if x == 'male' else 0)
teste['MaleCheck'] = teste.Sex.apply(lambda x: 1 if x == 'male' else 0)

In [16]:
# Eliminando a coluna Sex
treino = treino.drop('Sex', axis=1)
teste = teste.drop('Sex', axis=1)

In [9]:
# Importando o RobustScaler
from sklearn.preprocessing import RobustScaler

# Criando o scaler
transformer = RobustScaler().fit(treino[['Age', 'Fare']])

# Fazendo a transformação dos dados
treino[['Age', 'Fare']] = transformer.transform(treino[['Age', 'Fare']])

# Fazendo o mesmo para a base de teste
transformer = RobustScaler().fit(teste[['Age', 'Fare']])
teste[['Age', 'Fare']] = transformer.transform(teste[['Age', 'Fare']])

In [12]:
# Criando uma função para verificar se dois valores são vazios
def sozinho(a,b):
    if(a==0 and b==0):
        return 1
    else:
        return 0

# Aplicando essa função na base de treino
treino['Sozinho'] = treino.apply(lambda x: sozinho(x.SibSp, x.Parch), axis=1)
# Fazendo o mesmo para base de teste
teste['Sozinho'] = teste.apply(lambda x: sozinho(x.SibSp, x.Parch), axis=1)

In [13]:
# Criando uma nova coluna com o total de familiares a bordo
treino['Familiares'] = treino.SibSp + treino.Parch
teste['Familiares'] = teste.SibSp + teste.Parch

In [14]:
# Usando o OrdinalEncoder para a coluna Embarked
# Importando OrdinalEncoder
from sklearn.preprocessing import OrdinalEncoder
categorias = ['S', 'C', 'Q']

enc = OrdinalEncoder(categories=[categorias], dtype='int32')
# faznedo o fit com os dados
enc = enc.fit(treino[['Embarked']])
# Adicionando essa coluna na base de treino original
treino['Embarked'] = enc.transform(treino[['Embarked']])

# Adicionando na base de teste original
teste['Embarked'] = enc.transform(teste[['Embarked']])

In [17]:
# Visualizando a base de treino
treino.head(3)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Familiares,Sozinho
0,1,0,3,-0.592240,1,0,-0.312011,0,1,1,0
1,2,1,1,0.638529,1,0,2.461242,1,0,1,0
2,3,1,3,-0.284548,0,0,-0.282777,0,0,0,1


## Selecionando novos algoritmos de previsão
**Observação:** Vou manter o algoritmo de Regressão Logística por, anteriormente, ter tido o melhor resultado.

In [20]:
# Importando o train_test_split
from sklearn.model_selection import train_test_split

In [18]:
# Sepearando a base de treino em X e Y
X =  treino.drop(['PassengerId', 'Survived'], axis=1)
y = treino.Survived

In [21]:
# Separando em treino e validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.33, random_state=42)

### Regressão Logística

In [19]:
# Importando
from sklearn.linear_model import LogisticRegression

In [25]:
# Criando o classificador
clf_rl = LogisticRegression(random_state=42)

In [27]:
# Fazendo o fit com os dados
clf_rl = clf_rl.fit(X_train, y_train)

In [29]:
# Fazendo a previsão
y_pred_rl = clf_rl.predict(X_val)

### RandomForest

In [30]:
# Importação
from sklearn.ensemble import RandomForestClassifier

In [32]:
# Criando o classificador
clf_rf = RandomForestClassifier(random_state=42)

In [33]:
# Fazendo o fit com os dados
clf_rf = clf_rf.fit(X_train, y_train)

In [34]:
# Fazendo a previsão
y_pred_rf = clf_rf.predict(X_val)

### MLPClassifier (Redes Neurais)

In [35]:
# Importando
from sklearn.neural_network import MLPClassifier

In [36]:
# Criando o classificador
clf_mlp = MLPClassifier(random_state=42, max_iter=5000)

In [37]:
# Fazendo o fit com os dados
clf_mlp = clf_mlp.fit(X_train, y_train)

In [38]:
# Fazendo a previsão
y_pred_mlp = clf_mlp.predict(X_val)

## Avaliando os modelos

### Acurácia

In [39]:
# Importando
from sklearn.metrics import accuracy_score

In [41]:
# Para a Regressão Logística
accuracy_score(y_val, y_pred_rl)

0.8067796610169492

In [42]:
# Para o Random Forest
accuracy_score(y_val, y_pred_rf)

0.8067796610169492

In [43]:
# Para o MLPClassifier (Redes Neurais)
accuracy_score(y_val, y_pred_mlp)

0.8203389830508474

### Matriz de confusão

In [44]:
# Importando
from sklearn.metrics import confusion_matrix

In [45]:
# Para a Regressão Logística
confusion_matrix(y_val, y_pred_rl)

array([[152,  23],
       [ 34,  86]])

In [46]:
# Para o Random Forest
confusion_matrix(y_val, y_pred_rf)

array([[150,  25],
       [ 32,  88]])

In [47]:
# Para o MLPClassifier (Redes Neurais)
confusion_matrix(y_val, y_pred_mlp)

array([[158,  17],
       [ 36,  84]])

## Fazendo a previsão para os dados de teste

In [48]:
# Visualizando a base de teste
teste.head(3)

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Familiares,Sozinho
0,892,3,0.331562,0,0,-0.280670,2,1,0,1
1,893,3,1.311954,1,0,-0.315800,0,0,1,0
2,894,2,2.488424,0,0,-0.201943,2,1,0,1


In [49]:
# Para a base de teste ser igual a base de treino, precisamos eliminar a coluna de id
X_teste = teste.drop('PassengerId', axis=1)

In [50]:
# Utilizando o melhor modelo da base de teste
y_pred = clf_mlp.predict(X_teste)

In [51]:
# Criando uma nova coluna com a previsão na base de teste
teste['Survived'] = y_pred

In [52]:
# Selecionando apenas a coluna de Id e Survived para fazer o envio
base_envio = teste[['PassengerId', 'Survived']]

In [53]:
# Exportando para um csv
base_envio.to_csv('resultados4.csv', index=False)